# 00 — Coverage Discovery & 360° Equirectangular Rosette Strip (Zero Gemini)

This notebook shows that you can discover live Street View Insights snapshots, inspect spatial
coverage density and repeat-pass dates in BigQuery, and render a calibrated 360° equirectangular
strip and 7-camera contact sheet purely with deterministic geometry (**zero Gemini calls**).

> **Runtime & Cost Banner:** Expected runtime **<= 3 min** on Colab CPU · Gemini cost **$0.00**
> (0 calls) · Cold BigQuery scan **1.04 GB** (`coverage_sql`) + **1.83 GB** (`rosette_sql`) at
> **$6.25/TB** on-demand (~$0.02 first run, **0 B** when cached for up to 35 days).

## 1. Problem & Data Model

Each panoramic capture in `pano_observations_latest` is keyed by `capture_id` (7 frames `k = 0..6`,
where `0..5` form the horizontal ring ~60° apart and `6` points to the sky). The publishable
Street View key `pano_id` is nullable (~69% of captures in standard AOIs have `pano_id IS NULL`)
and is retained for attribution metadata alongside `map_url`.

**Division of labour.** BigQuery computes geohash-6 coverage cells (`coverage_sql`, 1.04 GB),
rosette sequences (`rosette_sql`, 1.83 GB), and drive LineStrings (`tracks_sql`, 1.43 GB); Python
unprojects and remaps the Kannala-Brandt fisheye frames into a 360° equirectangular strip.

**Terms of use & authentication.** Imagery comes from your Street View Insights dataset and is
governed by the terms of your Google Cloud agreement: do not redistribute frames or rendered crops
outside your organisation, keep the `"Imagery © Google"` attribution on every figure and map, and
do not persist raw imagery or `gcs_uri` values beyond short-lived working caches (<= 24 h for
frames, <= 35 days for snapshot-keyed metadata).

- **Colab:** calls `google.colab.auth.authenticate_user()`.
- **Local Jupyter:** run `gcloud auth application-default login` once.
- **Workstation helpers:** set `SVI_USE_GCLOUD_TOKEN=1` and `SVI_ECP_PROXY_URL` when required.
- **IAM roles needed:** `roles/bigquery.jobUser`, `roles/bigquery.dataViewer`,
  `roles/storage.objectViewer`, and `roles/aiplatform.user`.

In [ ]:
# Install the shared `svi_geo` helper package (geometry, sequencing, dedup, Gemini client).
# Dependency ranges in pyproject.toml are compatible with Colab preinstalled versions.
# In a clone of the repo the local copy is used: $SVI_GEO_LOCAL_PATH, or the first parent
# directory that contains street_view_insights/panoramic/svi_geo. Otherwise pip installs it
# from GitHub at SVI_GEO_REF.
import importlib.util
import os
import pathlib
import subprocess
import sys

# TODO(after-merge): set to "main" once svi_geo is merged. Override with $SVI_GEO_REF.
SVI_GEO_REF = os.environ.get("SVI_GEO_REF", "94b9530")


def find_local_svi_geo():
    if os.environ.get("SVI_GEO_LOCAL_PATH"):
        return os.environ["SVI_GEO_LOCAL_PATH"]
    here = pathlib.Path.cwd().resolve()
    for d in (here, *here.parents):
        if (d / "street_view_insights/panoramic/svi_geo" / "pyproject.toml").is_file():
            return str(d / "street_view_insights/panoramic/svi_geo")
    return None


LOCAL_SVI_GEO = find_local_svi_geo()
if importlib.util.find_spec("svi_geo") is None:
    if LOCAL_SVI_GEO:
        src = f"{LOCAL_SVI_GEO}[notebooks]"
    else:
        src = (
            "svi-geo[notebooks] @ git+https://github.com/googlemaps-samples/insights-samples"
            f"@{SVI_GEO_REF}#subdirectory=street_view_insights/panoramic/svi_geo"
        )
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", src])

In [ ]:
import json
import os

import matplotlib.pyplot as plt
from google.cloud import storage

from svi_geo import attribution, auth, config, data, images, maps, rosette

try:
    from google.colab import auth as colab_auth

    colab_auth.authenticate_user()
except ImportError:
    pass

PROJECT_ID = os.environ.get("PROJECT_ID", "YOUR_PROJECT_ID")
DATASET_ID = "imagery_insights___us"
GCS_BUCKET = os.environ.get("GCS_BUCKET", "")
SNAPSHOT_ID = None
INCLUDE_UNPUBLISHED_PANOS = True
LAT, LNG, RADIUS_M = 28.05047, -81.96015, 120.0
OUT_DIR = pathlib.Path(os.environ.get("OUT_DIR", "~/tmp/svi_qa/00_explore_coverage")).expanduser()
OUT_DIR.mkdir(parents=True, exist_ok=True)

settings = config.resolve_settings(PROJECT_ID, GCS_BUCKET, env=dict(os.environ), dataset=DATASET_ID)
PROJECT_ID, BUCKET = settings.project, settings.bucket
creds = auth.get_credentials()
TABLE = data.pano_table(PROJECT_ID, DATASET_ID)
SNAP_TABLE = f"{PROJECT_ID}.{DATASET_ID}.snapshots"
bq = data.QueryRunner(
    data.make_bigquery_client(PROJECT_ID, creds),
    allowed_tables=data.pano_tables(PROJECT_ID, DATASET_ID),
    cache_dir=data.DEFAULT_QUERY_CACHE,
)
fetcher = images.GcsImageFetcher(storage.Client(project=PROJECT_ID, credentials=creds))
intr = rosette.load_intrinsics()

In [ ]:
# Discover live SV_PANO snapshots (2 KB scan) and inspect geohash-6 coverage cells (1.04 GB).
snap_sql = data.snapshot_catalog_sql(SNAP_TABLE)
snap_df = bq.run(snap_sql, [])
cov_sql = data.coverage_sql(TABLE)
cov_params = data.coverage_params(
    lat_min=LAT - 0.003, lat_max=LAT + 0.003, lng_min=LNG - 0.003, lng_max=LNG + 0.003
)
cov_df = bq.run(cov_sql, cov_params, template_name="coverage_sql")

ros_sql = data.rosette_sql(TABLE)
ros_params = data.rosette_params(
    lat=LAT,
    lng=LNG,
    radius_m=RADIUS_M,
    snapshot_id=SNAPSHOT_ID,
    include_unpublished=INCLUDE_UNPUBLISHED_PANOS,
)
rosettes = bq.run(ros_sql, ros_params, template_name="rosette_sql")
snap_summary = data.summarize_snapshots(rosettes, snap_df)
print(snap_summary["summary_line"])
frames = data.frames_from_rosettes(rosettes, bucket=BUCKET)
cost = bq.last_cost()
snap_ids = ",".join(s[:8] for s in snap_summary["snapshot_ids"]) or "none"
null_cnt = int(rosettes["pano_id"].isna().sum())
print(
    f"snapshots={snap_ids} rosettes={len(rosettes)} null_pano_id={null_cnt} "
    f"sequences={rosettes['seq_id'].nunique()} bq_bytes={cost['bytes']} "
    f"bq_cached={int(cost['cached'])}"
)
print(f"coverage_cells={len(cov_df)}")

In [ ]:
# Worked pixel <-> ray <-> world example and 360 deg equirectangular strip for one rosette.
cid0 = str(rosettes.iloc[0]["capture_id"])
r0_frames = frames[frames["capture_id"] == cid0].sort_values("cam_k")
row0 = r0_frames.iloc[0].to_dict()
view0 = rosette.PerspectiveView(
    yaw_deg=float(row0["camera_pose"]["heading"]),
    pitch_deg=0.0,
    hfov_deg=70.0,
    width=1024,
    height=768,
)
az_deg, el_deg = view0.pixel_to_bearing(512.0, 384.0)
print(f"worked_example capture_id={cid0} centre_bearing=({az_deg:.2f}, {el_deg:.2f}) deg")

r0_rows = r0_frames.to_dict("records")
frames6 = {
    int(r["cam_k"]): images.decode(fetcher.fetch(r["gcs_uri"]), scale=0.5)
    for r in r0_rows
    if int(r["cam_k"]) < 6
}
strip_bgr = rosette.render_equirect_strip(
    frames6, intr, r0_rows, width=2048, pitch_range=(-25.0, 35.0)
)
b0 = rosette.view_black_fraction(intr, row0["camera_pose"], view0, int(row0["cam_k"]))
print(f"strips_rendered=1 black_fraction_max={b0:.4f}")
fig, ax = plt.subplots(figsize=(12, 3.2))
ax.imshow(strip_bgr[:, :, ::-1])
ax.set_title(f"360° Equirectangular Strip — {cid0}")
ax.axis("off")
attribution.save_figure(fig, OUT_DIR / "equirect_strip.png")
plt.close(fig)

In [ ]:
# Drive tracks map from WKT points and appendix metrics artefact.
map_path = OUT_DIR / "map.html"
m = maps.rosette_tracks_map(rosettes, out_path=map_path)
metrics_path = OUT_DIR / "metrics.json"
metrics_path.write_text(
    json.dumps(
        {
            "notebook": "00_explore_coverage",
            "rosettes": int(len(rosettes)),
            "null_pano_id": int(rosettes["pano_id"].isna().sum()),
            "coverage_cells": int(len(cov_df)),
        },
        indent=2,
    ),
    encoding="utf-8",
)
artefacts = len(list(OUT_DIR.glob("*")))
print(f"map_rendered=1 artefacts={artefacts} peak_rss_mb={images.peak_rss_mb():.0f}")

## Appendix: User-Owned Clustered Pano Index Pattern (O3, Doc-Only)

Analytics Hub linked datasets expose logical views and do not permit Materialized Views on the
linked dataset. Subscribers who run many local AOI queries per day can materialise a metadata-only
clustered table in their own dataset with a 35-day TTL (never storing `gcs_uri` or raw imagery):

```sql
CREATE OR REPLACE TABLE `my_project.my_dataset.svi_rosette_index`
CLUSTER BY geog
OPTIONS (expiration_timestamp = TIMESTAMP_ADD(CURRENT_TIMESTAMP(), INTERVAL 35 DAY))
AS
SELECT capture_id, snapshot_id, ANY_VALUE(pano_id) AS pano_id, MIN(capture_time) AS capture_time,
       ANY_VALUE(ST_GEOGPOINT(capture_location.longitude, capture_location.latitude)) AS geog,
       ANY_VALUE(map_url) AS map_url
FROM `my_project.imagery_insights___us.pano_observations_latest`
GROUP BY capture_id, snapshot_id;
```